In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from pathlib import Path
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest, confint_proportions_2indep
from statsmodels.stats.power import NormalIndPower
from utils import cleaning, summary, purchase_period


In [2]:
file_path = 'zomato_raw_data.csv'
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')


In [3]:
# The file in this repo is CSV text with an .xls name.
if not Path(file_path).exists():
    file_path = 'data/zomato_raw_data.xls'


In [4]:
first_period = 0
eligible_period = 2
outcome_period = 3
min_lift = 0.05
true_lift = 0.05
discount_pct = 0.10
alpha = 0.05
power = 0.80
seed = 42
n_sim = 1000


### Data Check

I first check whether the same users come back in different months.
The existing notebooks stay unchanged. The helpers in `utils.py` copy their cleaning,
session summary, first purchase month and period logic. Only unused comments were removed.


In [5]:
df = cleaning(file_path)


In [6]:
print(df.groupby('user_id')['session_id'].nunique().describe())


count    5000.000000
mean        1.114000
std         0.463732
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         8.000000
Name: session_id, dtype: float64


In [7]:
def data_check(df):
    session_count = df.groupby('user_id')['session_id'].nunique()
    purchase_months = df[df['revenue'] > 0].groupby('user_id')['month'].nunique()
    check_df = pd.DataFrame({
        'users': [df['user_id'].nunique()],
        'sessions': [df['session_id'].nunique()],
        'users_with_repeat_sessions': [(session_count > 1).sum()],
        'users_buying_in_2_months': [(purchase_months >= 2).sum()]
    })
    return check_df


In [8]:
check_df = data_check(df)


In [9]:
check_df


,users,sessions,users_with_repeat_sessions,users_buying_in_2_months
0,5000,5570,378,361


In [10]:
if df.groupby('user_id')['session_id'].nunique().max() == 1:
    print('WARNING: Users have only one session. Month-to-month retention is not valid with this data.')
    raise ValueError('Stop here. Ask for confirmation before continuing.')
if check_df['users_buying_in_2_months'].sum() == 0:
    raise ValueError('Stop here. There are no users with purchases in different months.')


<div class="alert alert-info">
<b>Observation:</b> There are 5,000 users and 5,570 sessions. 378 users have repeat sessions,
and 361 users purchased in at least two months, so the data passes this check.
</div>

**Hypothesis:**
- An offer may help some of these buyers come back again.
- These are synthetic users, so this check does not prove real customer behavior.


### Experiment Design

Business problem: users stop buying after the first few months. The cohort notebook suggests
a sharp drop by month 4. I will test an offer around month 3. This is an idea to test, not proof
that month 3 is the best time.

**Hypothesis:**
- Users who get an offer will buy again more often than users who get nothing.
- Acquisition month means first purchase month, as in the purchase cohorts.
- Eligible users purchased in period 0 and period 2. Send the offer at the start of period 3.
- Period 0 is the first purchase month. Period 3 is three calendar months later.
- Randomly split eligible users 50/50. Control gets nothing. Treatment gets the offer.
- Keep each user in one group and use a fixed seed.

Primary metric: `Repeat purchase rate = period-3 buyers / eligible users`.

Guardrail: `Revenue per eligible user after cost = (period-3 revenue - discount cost) / eligible users`.
Discount cost is 10% of all treatment revenue in period 3, including purchases that might happen anyway.
Control has no discount cost. This is revenue after the offer cost, not profit.

Decision rule: ship only if the lift is positive, the two-sided p-value is below 0.05,
and treatment revenue per user after cost is at least the control value.


In [11]:
summ_df = summary(df)


In [12]:
payment_df = purchase_period(summ_df)


In [13]:
def eligible_users(payment_df, summ_df, first_period, eligible_period, outcome_period):
    first_df = payment_df[payment_df['period'] == first_period][['user_id', 'a_month']].drop_duplicates()
    repeat_df = payment_df[payment_df['period'] == eligible_period][['user_id']].drop_duplicates()
    eligible_df = first_df.merge(repeat_df, on='user_id')

    # 1. Keep users whose outcome month ended before the last observed month.
    last_month = summ_df['month'].max()
    eligible_df = eligible_df[eligible_df['a_month'] + outcome_period < last_month].copy()

    outcome_df = payment_df[payment_df['period'] == outcome_period].groupby('user_id').agg(
        revenue=('revenue', 'sum')
    ).reset_index()
    eligible_df = eligible_df.merge(outcome_df, on='user_id', how='left')
    eligible_df['revenue'] = eligible_df['revenue'].fillna(0)
    eligible_df['purchased'] = (eligible_df['revenue'] > 0).astype(int)
    return eligible_df


In [14]:
if not 0 <= first_period < eligible_period < outcome_period:
    raise ValueError('Use increasing period numbers, starting at 0 or later.')
eligible_df = eligible_users(payment_df, summ_df, first_period, eligible_period, outcome_period)


In [15]:
eligible_df


,user_id,a_month,revenue,purchased
0,USR00026,2024-02,987.22,1
1,USR00093,2024-06,0.00,0
2,USR00252,2024-02,348.37,1
3,USR00262,2024-06,398.60,1
4,USR00276,2024-04,0.00,0
...,...,...,...,...
97,USR04856,2024-05,0.00,0
98,USR04878,2024-01,0.00,0
99,USR04887,2024-06,0.00,0
100,USR04918,2024-03,0.00,0


<div class="alert alert-info">
<b>Observation:</b> 147 users purchased in period 2. Only 102 have a full period-3 window before
the last observed month. The other 45 are left out, rather than counted as users who did not return.
</div>

The data ends on October 30, 2024, so October may be incomplete. I only use outcome months
before October. A real test should wait until every user's outcome month has ended.

**Hypothesis:**
- Checking a full month gives each eligible user the same chance to buy again.


### Baseline and Sample Size


In [16]:
def baseline(eligible_df):
    if len(eligible_df) == 0:
        raise ValueError('There are no eligible users with a full outcome month.')
    baseline_df = pd.DataFrame({
        'eligible_users': [len(eligible_df)],
        'repeat_buyers': [eligible_df['purchased'].sum()],
        'repeat_rate_pct': [eligible_df['purchased'].mean() * 100]
    }).round(2)
    return baseline_df


In [17]:
baseline_df = baseline(eligible_df)


In [18]:
baseline_df


,eligible_users,repeat_buyers,repeat_rate_pct
0,102,26,25.49


In [19]:
base_rate = eligible_df['purchased'].mean()
if not 0 < base_rate < 1:
    raise ValueError('A baseline between 0 and 1 is needed for this sample size plan.')
if not 0 < min_lift < 1 - base_rate or not 0 <= true_lift <= 1 - base_rate:
    raise ValueError('Choose lifts that keep the purchase rate between 0 and 1.')
if not 0 <= discount_pct <= 1:
    raise ValueError('Discount must be between 0 and 1.')


In [20]:
def sample_size(base_rate, lifts, alpha, power):
    size_rows = []
    for lift in lifts:
        effect = proportion_effectsize(base_rate + lift, base_rate)
        needed = NormalIndPower().solve_power(
            effect_size=effect, alpha=alpha, power=power,
            ratio=1, alternative='two-sided'
        )
        size_rows.append({
            'min_lift_pp': round(lift * 100, 2),
            'users_per_group': int(np.ceil(needed))
        })
    size_df = pd.DataFrame(size_rows)
    return size_df


In [21]:
size_df = sample_size(base_rate, [min_lift], alpha, power)


In [22]:
size_df


,min_lift_pp,users_per_group
0,5.0,1264


In [23]:
users_needed = int(size_df['users_per_group'].sum())
users_available = len(eligible_df)
print('Users needed per group:', users_needed)
print('Eligible users available in total:', users_available)
print('Users available per group:', users_available // 2)
print('Enough users:', users_available >= 2 * users_needed)


Users needed per group: 1264
Eligible users available in total: 102
Users available per group: 51
Enough users: False


<div class="alert alert-info">
<b>Observation:</b> The baseline is 25.49% (26 out of 102). A 5 percentage point lift
needs 1264 users per group, or 2528 in total. This dataset only gives 51 per group.
</div>

This dataset is too small for the planned test. In real life, I would wait for more eligible users
and their full outcome months. A test this small can miss an offer that helps users.
The baseline is for users who already bought in period 2, not all acquired users.

**Hypothesis:**
- A smaller lift needs more users to separate the offer's effect from chance.
- `Planned treatment rate = baseline rate + minimum lift`.


In [24]:
lifts = [0.01, 0.02, 0.03, 0.04, 0.05, 0.075, 0.10]
lifts = [lift for lift in lifts if base_rate + lift < 1]
curve_df = sample_size(base_rate, lifts, alpha, power)


In [25]:
fig = px.line(
    curve_df, x='min_lift_pp', y='users_per_group', markers=True,
    title='Users Needed Per Group by Minimum Lift',
    labels={'min_lift_pp': 'Minimum lift (percentage points)', 'users_per_group': 'Users per group'}
)
fig.show()


### SIMULATED Offer Test

This is a practice test. It does not show the effect of a real offer.
Control keeps the recorded period-3 purchase and revenue. Treatment purchases are drawn at
`baseline rate + true_lift`, with an assumed lift of 5 percentage points.

For each simulated treatment buyer, I assume the same period-3 spend: the average total period-3
revenue per buyer in the eligible data. Non-buyers have zero revenue. Then I subtract the discount
from all treatment revenue. This simple assumption does not measure how an offer changes spending.


In [26]:
def simulate(eligible_df, base_rate, true_lift, discount_pct, seed):
    sim_df = eligible_df.sample(frac=1, random_state=seed).reset_index(drop=True).copy()
    sim_df['group'] = 'treatment'
    sim_df.loc[sim_df.index < len(sim_df) // 2, 'group'] = 'control'
    treatment = sim_df['group'] == 'treatment'
    rng = np.random.default_rng(seed)
    sim_df.loc[treatment, 'purchased'] = rng.binomial(1, base_rate + true_lift, treatment.sum())

    buyer_revenue = eligible_df.loc[eligible_df['purchased'] == 1, 'revenue'].mean()
    sim_df.loc[treatment, 'revenue'] = sim_df.loc[treatment, 'purchased'] * buyer_revenue
    sim_df['discount_cost'] = 0.0
    sim_df.loc[treatment, 'discount_cost'] = sim_df.loc[treatment, 'revenue'] * discount_pct
    sim_df['revenue_after_cost'] = sim_df['revenue'] - sim_df['discount_cost']
    return sim_df


In [27]:
sim_df = simulate(eligible_df, base_rate, true_lift, discount_pct, seed)


In [28]:
def group_result(sim_df):
    group_df = sim_df.groupby('group').agg(
        users=('user_id', 'nunique'),
        buyers=('purchased', 'sum'),
        repeat_rate=('purchased', 'mean'),
        revenue_per_user=('revenue', 'mean'),
        discount_per_user=('discount_cost', 'mean'),
        revenue_after_cost_per_user=('revenue_after_cost', 'mean')
    )
    group_df['repeat_rate_pct'] = group_df['repeat_rate'].mul(100).round(2)
    group_df = group_df.drop(columns='repeat_rate')
    group_df.columns.name = 'SIMULATED'
    return group_df


In [29]:
group_df = group_result(sim_df)


In [30]:
group_df


SIMULATED,users,buyers,revenue_per_user,discount_per_user,revenue_after_cost_per_user,repeat_rate_pct
group,,,,,,
control,51,9,126.756275,0.000000,126.756275,17.65
treatment,51,19,262.128929,26.212893,235.916036,37.25


In [31]:
fig = px.bar(
    group_df.reset_index(), x='group', y='repeat_rate_pct', color='group',
    title='SIMULATED Period-3 Repeat Purchase Rate',
    labels={'group': 'Group', 'repeat_rate_pct': 'Repeat purchase rate (%)'}
)
fig.show()


In [32]:
def test_result(group_df, alpha):
    control_n = group_df.loc['control', 'users']
    treatment_n = group_df.loc['treatment', 'users']
    control_buyers = group_df.loc['control', 'buyers']
    treatment_buyers = group_df.loc['treatment', 'buyers']
    control_rate = control_buyers / control_n
    treatment_rate = treatment_buyers / treatment_n

    if control_buyers + treatment_buyers in [0, control_n + treatment_n]:
        p_value = 1.0
    else:
        z_stat, p_value = proportions_ztest(
            [treatment_buyers, control_buyers], [treatment_n, control_n], alternative='two-sided'
        )
    low, high = confint_proportions_2indep(
        treatment_buyers, treatment_n, control_buyers, control_n,
        compare='diff', method='newcomb', alpha=alpha
    )
    guardrail_ok = group_df.loc['treatment', 'revenue_after_cost_per_user'] >= group_df.loc['control', 'revenue_after_cost_per_user']
    ship = treatment_rate > control_rate and p_value < alpha and guardrail_ok
    result_df = pd.DataFrame({
        'control_rate_pct': [control_rate * 100],
        'treatment_rate_pct': [treatment_rate * 100],
        'lift_pp': [(treatment_rate - control_rate) * 100],
        'p_value': [p_value],
        'ci_low_pp': [low * 100],
        'ci_high_pp': [high * 100],
        'guardrail_ok': [guardrail_ok],
        'decision': ['SHIP' if ship else 'DO NOT SHIP']
    })
    pct_cols = ['control_rate_pct', 'treatment_rate_pct', 'lift_pp', 'ci_low_pp', 'ci_high_pp']
    result_df[pct_cols] = result_df[pct_cols].round(2)
    result_df.columns.name = 'SIMULATED'
    return result_df


In [33]:
result_df = test_result(group_df, alpha)


In [34]:
result_df


SIMULATED,control_rate_pct,treatment_rate_pct,lift_pp,p_value,ci_low_pp,ci_high_pp,guardrail_ok,decision
0,17.65,37.25,19.61,0.026505,2.25,35.53,True,SHIP


In [35]:
print('SIMULATED decision:', result_df.loc[0, 'decision'])


SIMULATED decision: SHIP


<div class="alert alert-info">
<b>Observation:</b> SIMULATED control is 17.65% and treatment is 37.25%.
The lift is 19.61 percentage points, with p = 0.0265 and a 95% interval from
2.25 to 35.53 points. The decision is SHIP.
</div>

SIMULATED revenue per eligible user after cost is 126.76 for control and
235.92 for treatment. The guardrail passes.
The decision uses the simple rule above. A real rollout needs a properly sized real test.

**Hypothesis:**
- The observed lift can differ from the assumed 5 points because the groups are small.


### SIMULATED Detection Over 1,000 Tests

Each repeat uses a new fixed seed for the split and treatment draw. Control keeps its recorded
outcomes each time. I count positive lifts with p < 0.05. This is detection, not the share of tests
that pass the revenue guardrail. It is a check using this fixed dataset, not a full estimate of
power across new populations.


In [36]:
def repeat_simulation(eligible_df, base_rate, true_lift, discount_pct, seed, n_sim, alpha, power):
    sim_rows = []
    for run in range(n_sim):
        sim_df = simulate(eligible_df, base_rate, true_lift, discount_pct, seed + run + 1)
        counts = sim_df.groupby('group')['purchased'].agg(['sum', 'count'])
        control_rate = counts.loc['control', 'sum'] / counts.loc['control', 'count']
        treatment_rate = counts.loc['treatment', 'sum'] / counts.loc['treatment', 'count']
        total_buyers = counts['sum'].sum()
        if total_buyers in [0, counts['count'].sum()]:
            p_value = 1.0
        else:
            z_stat, p_value = proportions_ztest(
                counts.loc[['treatment', 'control'], 'sum'],
                counts.loc[['treatment', 'control'], 'count'], alternative='two-sided'
            )
        sim_rows.append({'detected': treatment_rate > control_rate and p_value < alpha})
    runs_df = pd.DataFrame(sim_rows)
    detection_df = pd.DataFrame({
        'tests': [n_sim],
        'detected_tests': [runs_df['detected'].sum()],
        'detection_pct': [round(runs_df['detected'].mean() * 100, 2)],
        'target_pct': [round(power * 100, 2)]
    })
    detection_df.columns.name = 'SIMULATED'
    return detection_df


In [37]:
detection_df = repeat_simulation(eligible_df, base_rate, true_lift, discount_pct, seed, n_sim, alpha, power)


In [38]:
detection_df


SIMULATED,tests,detected_tests,detection_pct,target_pct
0,1000,55,5.5,80.0


<div class="alert alert-info">
<b>Observation:</b> SIMULATED detection is 5.50% over 1,000 tests, compared with
the 80% target. The small dataset often misses the assumed lift.
</div>

**Hypothesis:**
- More eligible users should make the test better at detecting a real 5 point lift.


### What These Numbers Mean

**Why split users randomly?**
- It gives each user the same chance of getting the offer. This helps make the groups similar,
  although small groups can still differ by chance.

**What does the p-value mean?**
- If the offer had no effect, it is the chance of seeing a difference at least this large in
  either direction under the test's assumptions. It is not the chance that the offer has no effect.

**What does the confidence interval mean?**
- It shows a range of lifts that fit the data. If we repeated this method on new samples,
  about 95% of the intervals would cover the true lift. This interval uses the Newcombe method.
- An interval that includes zero leaves room for no improvement.

**Why might a significant result still be a bad offer?**
- More people may buy, but discounts can cost more than the extra revenue. That is why I also
  check revenue per eligible user after the discount cost. This guardrail is a simple comparison,
  and does not prove that revenue will never fall.

### Limits

- The source data is Faker-generated. These are not real Zomato customers.
- Treatment purchases, the 5 point lift and treatment spend are assumed. No offer was sent.
- This tests one offer in one window. It does not compare different months or offers.
- The baseline comes from a small group and may change in a larger real test.
- Keep the split fixed in a real test and check results once the planned sample and months are complete.

Method references: [sample size](https://www.statsmodels.org/stable/generated/statsmodels.stats.power.NormalIndPower.solve_power.html),
[two-proportion test](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportions_ztest.html),
[confidence interval](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.confint_proportions_2indep.html).


### 30-Second Summary


In [39]:
print('Baseline repeat purchase rate:', round(base_rate * 100, 2), '%')
print('Users needed per group:', users_needed)
print('Eligible users available:', users_available)
print('SIMULATED lift:', result_df.loc[0, 'lift_pp'], 'percentage points')
print('SIMULATED p-value:', round(result_df.loc[0, 'p_value'], 4))
print('SIMULATED decision:', result_df.loc[0, 'decision'])


Baseline repeat purchase rate: 25.49 %
Users needed per group: 1264
Eligible users available: 102
SIMULATED lift: 19.61 percentage points
SIMULATED p-value: 0.0265
SIMULATED decision: SHIP
